# PANDORA data exploration

Explore the filtered comments and profiles with Big Five scores. Run the cells in order after creating `filtered_comments.csv` and `big5_profiles_filtered.csv`. Change the paths below if needed.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA = Path("../data")  # From the project's src folder
COMMENTS = DATA / "filtered_comments.csv"
PROFILES = DATA / "big5_profiles_filtered.csv"

TRAITS = ["agreeableness", "openness", "conscientiousness",
          "extraversion", "neuroticism"]
PERSONALITY_SUBREDDITS = {
    "mbti", "intp", "infj", "infp", "intj", "entp", "enfp", "enfj", "entj",
    "istj", "isfj", "istp", "isfp", "estj", "esfj", "estp", "esfp",
    "enneagram", "myersbriggs", "personalitycafe", "mbtiplus",
}

## 1. Load the data and check its size

In [ ]:
comments = pd.read_csv(COMMENTS)
profiles = pd.read_csv(PROFILES)

print(f"Comments: {len(comments):,}")
print(f"Users with Big Five scores: {len(profiles):,}")
print(f"Users with comments: {comments['author'].nunique():,}")

## 2. Check the number of comments per user

In [ ]:
comments_per_user = comments.groupby("author").size()
display(comments_per_user.describe())

few_comments = comments_per_user < 20
share = few_comments.mean() if len(few_comments) else 0
print(f"Users with fewer than 20 comments: {few_comments.sum():,} ({share:.1%})")

## 3. Check comment length

In [ ]:
if "word_count" in comments.columns:
    word_counts = comments["word_count"]
    display(word_counts.describe())
    print(f"Comments with fewer than 5 words: {(word_counts < 5).sum():,}")
else:
    print("Comment length is unavailable because the word_count column is missing.")

## 4. Explore subreddit activity

Participation in personality communities can reveal information related to the prediction targets. This check flags potential label leakage in subreddit features.

In [ ]:
subreddit_counts = comments["subreddit"].value_counts()
print(f"Unique subreddits: {len(subreddit_counts):,}")
print("Top 15 subreddits by comment count")
display(subreddit_counts.head(15))

subreddit_names = comments["subreddit"].astype("string").str.lower()
personality_comments = subreddit_names.isin(PERSONALITY_SUBREDDITS)
found_subreddits = sorted(subreddit_names[personality_comments].unique())
share = personality_comments.mean() if len(comments) else 0

print(f"Personality-related subreddits found: {found_subreddits}")
print(f"Comments in these subreddits: {personality_comments.sum():,} "
      f"({share:.1%} of all comments)")

## 5. Check for missing values and repeated comments

In [ ]:
display(comments.isna().sum().rename("missing_values"))

deleted_comments = comments["body"].isin(["[deleted]", "[removed]"]).sum()
repeated_comments = comments.duplicated(subset=["author", "body"]).sum()
print(f"Deleted or removed comments: {deleted_comments:,}")
print(f"Repeated author and text pairs: {repeated_comments:,}")

## 6. Summarize the Big Five scores

In [ ]:
display(profiles[TRAITS].describe())

## 7. Check the date range

In [ ]:
if "created_utc" in comments.columns:
    dates = pd.to_datetime(comments["created_utc"], unit="s", utc=True, errors="coerce")
    print(f"Earliest comment (UTC): {dates.min()}")
    print(f"Latest comment (UTC): {dates.max()}")
else:
    print("The date range is unavailable because the created_utc column is missing.")